# Objective

Built a chatbot to call tools using yfinance library, and incorporate mcp server and client.

Here we also extend the MCP chatbot capabilities by making it connect to other MCP servers:
1. Fetch server
2. Filesystem server

Further, we include prompts and resources in the server.

## Tool Functions

In [7]:
%%writefile stock_mcp_server.py
import os
import pandas as pd
import yfinance as yf

from mcp.server.fastmcp import FastMCP

# Initialize FastMCP server
mcp = FastMCP("stock_mcp")

DATA_DIR = "data"

@mcp.tool()
def get_price(stock_code: str) -> float:
    """
    Get stock data from Yahoo Finance.

    Args:
        stock_code (str): The stock code to fetch data for.

    Returns:
        float: The current stock price.
    """

    stk = yf.Ticker(stock_code)

    try:
        stk_price = stk.info['currentPrice']
    except:
        try:
            stk_price = stk.info['previousClose']
        except:
            print("Error!!!!!! Unable to get stk_price.")

    return stk_price

@mcp.tool()
def get_analyst_price_target(stock_code: str) -> float:
    """
    Get analyst price targets from Yahoo Finance.

    Args:
        stock_code (str): The stock code to fetch data for.

    Returns:
        float: The mean analyst price target.
    """
    stk = yf.Ticker(stock_code)

    try:
        price_target = stk.analyst_price_targets['mean']
    except:
        print("Error!!!!!! Unable to get analyst price targets.")

    return price_target

@mcp.resource("data://folders")
def get_available_folders() -> str:
    """
    List all available stock folders in the data directory.
    
    This resource provides a simple list of all available stock folders.
    """
    folders = []
    
    # Get all topic directories
    if os.path.exists(DATA_DIR):
        for topic_dir in os.listdir(DATA_DIR):
            topic_path = os.path.join(DATA_DIR, topic_dir)
            if os.path.isdir(topic_path):
                financials_file = os.path.join(topic_path, "financials.csv")
                if os.path.exists(financials_file):
                    folders.append(topic_dir)
    
    # Create a simple markdown list
    content = "# Available Stocks\n\n"
    if folders:
        for folder in folders:
            content += f"- {folder}\n"
        content += f"\nFor example, use @{folder} to access financial data about this particular stock.\n"
    else:
        content += "No topics found.\n"
    
    return content

@mcp.resource("data://{stock}")
def get_stock_data(stock: str) -> str:
    """
    Get detailed information about the stock.
    
    Args:
        topic: The stock to retrieve financial info for
    """
    stock_dir = stock.upper().replace(" ", "_")
    stock_file = os.path.join(DATA_DIR, stock_dir, "financials.csv")
    
    if not os.path.exists(stock_file):
        return f"# No financial data found for stock: {stock}\n\n"
    
    try:
        # Read the CSV file (handles commas in numbers)
        df = pd.read_csv(stock_file, thousands=",")

        # Convert to markdown
        markdown_table = df.to_markdown(index=False)
        
        return markdown_table
    except Exception as e:
        return f"# Error reading stock data for {stock}\n\nThe financials.csv file is corrupted."

@mcp.prompt()
def generate_stock_analysis_prompt(stock: str) -> str:
    """
    Generate a prompt for Claude to find and discuss stock analysis.
    """
    prompt = f"""You are a financial analyst specializing in stock market investments.
        Your task is to research and analyze the stock '{stock}' for potential investment in 2025. 

        Follow these instructions:
        1. First, retrieve the financial data for the stock '{stock}' using the provided resource 'data://{stock}'.

        2. Retrieve the current stock price using the tool 'get_price' with the stock code '{stock}', 
        as well as the analyst price target using the tool 'get_analyst_price_target'.

        3. Next, fetch https://www.google.com/search?q={stock}+stock+analysis+2025+buy+reasons, 
        do a summary including the data retrieved from the resources and tools, 

        4. Organize your findings in a clear, structured format with headings and bullet points for easy readability.

        5. At the end of your response, provide a BUY, SELL, or HOLD recommendation for the stock based on your analysis.

        6. Write the overall summary and your BUY, SELL or HOLD recommendation to a file '{stock}.txt'

    Remember to output a BUY, SELL, or HOLD recommendation for the stock at the end of your analysis! 
    """

    return prompt

if __name__ == "__main__":
    mcp.run(transport='stdio')

Overwriting stock_mcp_server.py


## Chatbot Code

The chatbot handles the user's queries one by one, but it does not persist memory across the queries.

In [8]:
%%writefile stock_mcp_client_prompt_resource.py

from dotenv import load_dotenv
from anthropic import Anthropic
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from contextlib import AsyncExitStack
import json
import asyncio
import nest_asyncio

nest_asyncio.apply()

load_dotenv()

class MCP_ChatBot:
    def __init__(self):
        self.exit_stack = AsyncExitStack()
        self.anthropic = Anthropic()
        # Tools list required for Anthropic API
        self.available_tools = []
        # Prompts list for quick display 
        self.available_prompts = []
        # Sessions dict maps tool/prompt names or resource URIs to MCP client sessions
        self.sessions = {}

    async def connect_to_server(self, server_name, server_config):
        try:
            server_params = StdioServerParameters(**server_config)
            stdio_transport = await self.exit_stack.enter_async_context(
                stdio_client(server_params)
            )
            read, write = stdio_transport
            session = await self.exit_stack.enter_async_context(
                ClientSession(read, write)
            )
            await session.initialize()
            
            
            try:
                # List available tools
                response = await session.list_tools()
                for tool in response.tools:
                    self.sessions[tool.name] = session
                    self.available_tools.append({
                        "name": tool.name,
                        "description": tool.description,
                        "input_schema": tool.inputSchema
                    })
            
                # List available prompts
                prompts_response = await session.list_prompts()
                if prompts_response and prompts_response.prompts:
                    for prompt in prompts_response.prompts:
                        self.sessions[prompt.name] = session
                        self.available_prompts.append({
                            "name": prompt.name,
                            "description": prompt.description,
                            "arguments": prompt.arguments
                        })
                # List available resources
                resources_response = await session.list_resources()
                if resources_response and resources_response.resources:
                    for resource in resources_response.resources:
                        resource_uri = str(resource.uri)
                        self.sessions[resource_uri] = session
            
            except Exception as e:
                print(f"Error {e}")
                
        except Exception as e:
            print(f"Error connecting to {server_name}: {e}")

    async def connect_to_servers(self):
        try:
            with open("server_config.json", "r") as file:
                data = json.load(file)
            servers = data.get("mcpServers", {})
            for server_name, server_config in servers.items():
                await self.connect_to_server(server_name, server_config)
        except Exception as e:
            print(f"Error loading server config: {e}")
            raise
    
    async def process_query(self, query):
    
        messages = [{'role': 'user', 'content': query}]
        
        response = self.anthropic.messages.create(max_tokens=1024,
                                                  model="claude-sonnet-4-20250514", 
                                                  tools=self.available_tools, # tools exposed to the LLM
                                                  messages=messages)
        
        process_query = True
        while process_query:
            # Collect ALL content from assistant's response first
            assistant_content = []
            tool_results = []
            
            for content in response.content:
                if content.type == 'text':
                    print(content.text)
                    assistant_content.append(content)
                    
                elif content.type == 'tool_use':
                    assistant_content.append(content)
                    
                    tool_id = content.id
                    tool_args = content.input
                    tool_name = content.name
                    print(f"Calling tool {tool_name} with args {tool_args}")
                    
                    # tool invocation through the client session
                    # session = self.tool_to_session[tool_name] # new
                    session = self.sessions.get(content.name)
                    # print('session for tool ', tool_name, ' = ', session)
                    result = await session.call_tool(tool_name, arguments=tool_args)
                    # print('result of call_tool = ', result)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": tool_id,
                        "content": result.content
                    })
                    
            
            # Now append the assistant message ONCE with all content
            messages.append({'role': 'assistant', 'content': assistant_content})
            
            # If there were tool uses, add tool results and continue
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
                # print('messages after appending tool results = ', messages)
                print('len(messages) after appending tool results = ', len(messages))
                response = self.anthropic.messages.create(max_tokens=1024,
                                                          model="claude-sonnet-4-20250514", 
                                                          tools=self.available_tools,
                                                          messages=messages)
                                                                 
                # Check if we're done
                if len(response.content) == 1 and response.content[0].type == "text":
                    print(response.content[0].text)
                    process_query = False
            else:
                # No tool uses, we're done
                process_query = False 
    

    async def get_resource(self, resource_uri):
        session = self.sessions.get(resource_uri)
        
        # Fallback for data URIs - try any data resource session
        if not session and resource_uri.startswith("data://"):
            for uri, sess in self.sessions.items():
                if uri.startswith("data://"):
                    session = sess
                    break
            
        if not session:
            print(f"Resource '{resource_uri}' not found.")
            return
        
        try:
            result = await session.read_resource(uri=resource_uri)
            if result and result.contents:
                print(f"\nResource: {resource_uri}")
                print("Content:")
                print(result.contents[0].text)
            else:
                print("No content available.")
        except Exception as e:
            print(f"Error: {e}")
    
    async def list_prompts(self):
        """List all available prompts."""
        if not self.available_prompts:
            print("No prompts available.")
            return
        
        print("\nAvailable prompts:")
        for prompt in self.available_prompts:
            print(f"- {prompt['name']}: {prompt['description']}")
            if prompt['arguments']:
                print(f"  Arguments:")
                for arg in prompt['arguments']:
                    arg_name = arg.name if hasattr(arg, 'name') else arg.get('name', '')
                    print(f"    - {arg_name}")
    
    async def execute_prompt(self, prompt_name, args):
        """Execute a prompt with the given arguments."""
        session = self.sessions.get(prompt_name)
        if not session:
            print(f"Prompt '{prompt_name}' not found.")
            return
        
        try:
            result = await session.get_prompt(prompt_name, arguments=args)
            if result and result.messages:
                prompt_content = result.messages[0].content
                
                # Extract text from content (handles different formats)
                if isinstance(prompt_content, str):
                    text = prompt_content
                elif hasattr(prompt_content, 'text'):
                    text = prompt_content.text
                else:
                    # Handle list of content items
                    text = " ".join(item.text if hasattr(item, 'text') else str(item) 
                                  for item in prompt_content)
                
                print(f"\nExecuting prompt '{prompt_name}'...")
                await self.process_query(text)
        except Exception as e:
            print(f"Error: {e}")
    
    async def chat_loop(self):
        print("\nMCP Chatbot Started!")
        print("Type your queries or 'quit' to exit.")
        print("Use @stocks to see available stocks")
        print("Use /prompts to list available prompts")
        print("Use /prompt <name_of_prompt> <arg=value> to execute a prompt")
        
        while True:
            try:
                query = input("\nQuery: ").strip()
                if not query:
                    continue
        
                if query.lower() == 'quit':
                    break
                
                # Check for @resource syntax first
                if query.startswith('@'):
                    # Remove @ sign  
                    topic = query[1:]
                    if topic == "stocks":
                        resource_uri = "data://folders"
                    else:
                        resource_uri = f"data://{topic}"
                    await self.get_resource(resource_uri)
                    continue
                
                # Check for /command syntax
                if query.startswith('/'):
                    parts = query.split()
                    command = parts[0].lower()
                    
                    if command == '/prompts':
                        await self.list_prompts()
                    elif command == '/prompt':
                        if len(parts) < 2:
                            print("Usage: /prompt <name_of_prompt> <arg=value>")
                            continue
                        
                        prompt_name = parts[1]
                        args = {}
                        
                        # Parse arguments
                        for arg in parts[2:]:
                            if '=' in arg:
                                key, value = arg.split('=', 1)
                                args[key] = value
                        
                        await self.execute_prompt(prompt_name, args)
                    else:
                        print(f"Unknown command: {command}")
                    continue
                
                await self.process_query(query)
                    
            except Exception as e:
                print(f"\nError: {str(e)}")
    
    async def cleanup(self):
        await self.exit_stack.aclose()


async def main():
    chatbot = MCP_ChatBot()
    try:
        await chatbot.connect_to_servers()
        await chatbot.chat_loop()
    finally:
        await chatbot.cleanup()


if __name__ == "__main__":
    asyncio.run(main())

Overwriting stock_mcp_client_prompt_resource.py


In [ ]:
# Try with query:
# - @stocks
# - /prompts
# - /prompt generate_stock_analysis_prompt stock=GOOG
# - Give me the analyst targets for Tesla
# - Give me the current price for Tesla

# Resources

- [Quick Start for Client Developpers](https://modelcontextprotocol.io/quickstart/client)
- [Writing MCP client](https://github.com/modelcontextprotocol/python-sdk/blob/main/examples/clients/simple-chatbot/mcp_simple_chatbot/main.py)
- [Another mcp chatbot example](https://github.com/modelcontextprotocol/python-sdk/blob/main/examples/clients/simple-chatbot/mcp_simple_chatbot/main.py)